# 03 — Generate automatic GFAP proposals and manage annotations

This notebook creates the first automatic GFAP segmentation when no trained astrocyte checkpoint exists. It uses the separated GFAP channel and stores the result as a `pseudo` annotation outside the human-annotation directories.

Important distinction:

- `pseudo`: automatic proposal; excluded from supervised training by default.
- `seed`, `corrected`, `reviewed`: human-validated targets; included in training by default.

The automatic mask is useful immediately for QC and exploratory analysis. It is not silently presented as ground truth.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

## User settings

The defaults reproduce the tested `7d_453` proposal. Set `IMAGE_ID=None` to inspect the first generated row. `OVERWRITE_AUTOMATIC=True` is safe for derived pseudo artifacts; it never overwrites raw images or human annotations.

In [ ]:
SOURCE_MANIFEST = Path('data/metadata/manifest.csv')
PSEUDO_DIRECTORY = Path('outputs/pseudo_labels')
PSEUDO_MANIFEST = PSEUDO_DIRECTORY / 'manifest.csv'
IMAGE_ID = None
RUN_BOOTSTRAP = True
OVERWRITE_AUTOMATIC = True

GAUSSIAN_SIGMA = 0.8
LOW_THRESHOLD_RATIO = 0.4
HIGH_THRESHOLD_SCALE = 1.0
MIN_COMPONENT_AREA = 24
MAX_HOLE_AREA = 24

## Verify preprocessing prerequisites

Notebook 01 must have filled `gfap_channel`, `dapi_channel`, and the active nucleus-label path. The nucleus input is not required by the heuristic GFAP detector itself, but it is required by the later U-Net pipeline.

In [ ]:
from astroseg.io import load_manifest

source_manifest = load_manifest(SOURCE_MANIFEST)
required_values = ['gfap_channel', 'dapi_channel', 'cellpose_mask_path']
missing = {column: source_manifest.loc[source_manifest[column].str.strip() == '', 'image_id'].tolist() for column in required_values}
missing = {column: ids for column, ids in missing.items() if ids}
if missing:
    raise ValueError(f'Preparation is incomplete: {missing}. Run notebook 01 first.')
print(f'{len(source_manifest)} prepared image(s) are available.')
source_manifest[['image_id', 'gfap_channel', 'dapi_channel', 'cellpose_mask_path', 'annotation_status']]

## Generate automatic GFAP pseudo labels

The production script normalizes GFAP, smooths noise, finds strong signal using Otsu, and keeps weaker pixels only when connected to strong signal. This preserves many thin processes while limiting isolated faint background.

In [ ]:
if RUN_BOOTSTRAP:
    command = [
        sys.executable, 'scripts/generate_bootstrap_pseudo_labels.py',
        '--manifest', str(SOURCE_MANIFEST),
        '--output-dir', str(PSEUDO_DIRECTORY),
        '--output-manifest', str(PSEUDO_MANIFEST),
        '--gaussian-sigma', str(GAUSSIAN_SIGMA),
        '--low-threshold-ratio', str(LOW_THRESHOLD_RATIO),
        '--high-threshold-scale', str(HIGH_THRESHOLD_SCALE),
        '--min-component-area', str(MIN_COMPONENT_AREA),
        '--max-hole-area', str(MAX_HOLE_AREA),
    ]
    if OVERWRITE_AUTOMATIC:
        command.append('--overwrite')
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Bootstrap skipped because RUN_BOOTSTRAP=False')

pseudo_manifest = load_manifest(PSEUDO_MANIFEST)
pseudo_manifest[['image_id', 'annotation_path', 'annotation_status', 'annotation_source', 'review_status']]

In [ ]:
import pandas as pd

report_path = PSEUDO_DIRECTORY / 'bootstrap_report.csv'
report = pd.read_csv(report_path)
report

## Inspect one proposal

Red overlay pixels are included in the automatic GFAP mask. The foreground score is an uncalibrated heuristic confidence surface, not a neural-network probability.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tifffile

from astroseg.io import get_channel, load_ome_tiff
from astroseg.preprocessing import percentile_normalize

selected_id = IMAGE_ID or str(pseudo_manifest.iloc[0]['image_id'])
matches = pseudo_manifest.loc[pseudo_manifest['image_id'] == selected_id]
if len(matches) != 1:
    raise ValueError(f'IMAGE_ID must identify one pseudo-manifest row; found {len(matches)}')
row = matches.iloc[0]

def resolve_manifest_path(value, manifest_path=PSEUDO_MANIFEST):
    path = Path(str(value))
    for candidate in (path, manifest_path.parent / path):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(value)

microscopy = load_ome_tiff(resolve_manifest_path(row['path']))
gfap = get_channel(microscopy, str(row['gfap_channel']))
mask = tifffile.imread(resolve_manifest_path(row['annotation_path']))
probabilities = np.load(PSEUDO_DIRECTORY / 'probabilities' / f'{selected_id}.npy', allow_pickle=False)
overlay_path = PSEUDO_DIRECTORY / 'overlays' / f'{selected_id}.png'
overlay = plt.imread(overlay_path)
print('Mask shape / values:', mask.shape, np.unique(mask).tolist())
print('Foreground fraction:', float((mask > 0).mean()))

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(14, 12))
axes[0, 0].imshow(percentile_normalize(gfap), cmap='gray')
axes[0, 0].set_title('GFAP')
axes[0, 1].imshow(mask, cmap='gray')
axes[0, 1].set_title('Automatic binary pseudo mask')
axes[1, 0].imshow(probabilities[1], cmap='viridis', vmin=0, vmax=1)
axes[1, 0].set_title('Heuristic foreground score (uncalibrated)')
axes[1, 1].imshow(overlay)
axes[1, 1].set_title('Saved QC overlay')
for axis in axes.flat:
    axis.axis('off')
figure.suptitle(selected_id)
figure.tight_layout()
plt.show()

## Optional parameter comparison

Lower `low_threshold_ratio` includes more faint connected structure; a higher value is more conservative. This comparison is read-only and does not overwrite saved artifacts. Select one dataset-level setting rather than tuning every image independently.

In [ ]:
from astroseg.preprocessing import detect_gfap_bootstrap_mask

comparison_ratios = [0.3, 0.4, 0.5, 0.6]
figure, axes = plt.subplots(2, 2, figsize=(14, 14))
for axis, ratio in zip(axes.flat, comparison_ratios):
    result = detect_gfap_bootstrap_mask(
        gfap,
        gaussian_sigma=GAUSSIAN_SIGMA,
        low_threshold_ratio=ratio,
        high_threshold_scale=HIGH_THRESHOLD_SCALE,
        min_component_area=MIN_COMPONENT_AREA,
        max_hole_area=MAX_HOLE_AREA,
    )
    axis.imshow(percentile_normalize(gfap), cmap='gray')
    colored = np.zeros((*result.mask.shape, 4), dtype=np.float32)
    colored[..., 0] = 1.0
    colored[..., 3] = result.mask * 0.35
    axis.imshow(colored)
    axis.set_title(f'ratio={ratio:.1f}; foreground={result.foreground_fraction:.1%}')
    axis.axis('off')
figure.tight_layout()
plt.show()

## Optional: import a human-corrected version

If a pseudo mask is corrected in Fiji, Napari, or another annotation tool, save it with the original image dimensions. Then set the values below and enable the import cell. The importer archives the original export, creates a validated binary target and QC overlay, and writes a new manifest.

Leave `IMPORT_CORRECTED_MASK=False` for the fully automatic exploratory workflow.

In [ ]:
IMPORT_CORRECTED_MASK = False
CORRECTED_IMAGE_ID = selected_id
CORRECTED_MASK_PATH = None  # Example: Path('exports/7d_453_corrected.tiff')
ANNOTATOR = ''
CORRECTED_MANIFEST = Path('data/metadata/manifest_corrected.csv')

In [ ]:
if IMPORT_CORRECTED_MASK:
    if CORRECTED_MASK_PATH is None or not Path(CORRECTED_MASK_PATH).is_file():
        raise FileNotFoundError('Set CORRECTED_MASK_PATH to an existing aligned mask.')
    pairs_path = Path('data/metadata/corrected_annotation_pairs.csv')
    pd.DataFrame([{'image_id': CORRECTED_IMAGE_ID, 'mask_path': str(CORRECTED_MASK_PATH)}]).to_csv(pairs_path, index=False)
    command = [
        sys.executable, 'scripts/import_existing_annotations.py',
        '--manifest', str(PSEUDO_MANIFEST),
        '--pairs-csv', str(pairs_path),
        '--output-dir', 'data/annotations',
        '--output-manifest', str(CORRECTED_MANIFEST),
        '--status', 'corrected',
        '--source', 'manual_pseudo_correction',
        '--annotator', ANNOTATOR,
        '--overwrite',
    ]
    subprocess.run(command, check=True)
    print('Corrected manifest:', CORRECTED_MANIFEST)
else:
    print('Human-correction import skipped. Automatic pseudo artifacts remain separate.')

## Completion checklist

For the automatic workflow, confirm that the overlay follows meaningful GFAP-positive structures and that the generated manifest says `pseudo/pending`. Do not point supervised training at this manifest unless using pseudo targets is an explicit experimental decision.

Continue with notebook 04. It always runs a synthetic training smoke test and enables real training/evaluation only after it detects valid human annotations and splits.